# DX 704 Week 11 Project

In this project, you will develop and test prompts asking a language model to classify text from a home services query and match it to an appropriate category of home services.

The full project description and a template notebook are available on GitHub: [Project 11 Materials](https://github.com/bu-cds-dx704/dx704-project-11).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1 : Design a Short Prompt

The provided file "queries.txt" contains sample text from requests by homeowners by email or phone.
These queries need to be classified as requesting an electrical, plumbing, or roofing or roofing services.
The provided file has columns query_id, query, and target_category.
Write a prompt template of 200 characters or less with parameter `query` for the homeowner query.
Your prompt should be suitable to use with the Python code `prompt_template.format(query=query)`.
Test your prompt with the model `gpt-6-luna` and suitable parsing code.

In [1]:
%pip install -U "openai>=3.26.0"

Note: you may need to restart the kernel to use updated packages.


In [3]:
# from google.colab import userdata     # running directly on computer
from openai import OpenAI

In [4]:
# client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

# accessing API key from .env instead of Google Colab
import os
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [5]:
response = client.responses.create(
    model="gpt-6-luna",
    input="How much wood would a woodchuck chuck if a wood chuck could chuck wood?"
)

print(response.output_text)

As much as a woodchuck could chuck—if a woodchuck could chuck wood. 🪵🐿️


In [9]:
# testing one to see output

prompt       = "Categorize homeowner's request/issue by required service type to resolve them. Explicitly respond with the best service out of ONLY these three options: electrical, plumbing, roofing.\nQuery: {query}"
sample_query = "There seems to be water pooling on part of my ceiling on the top floor."

prompt_input = prompt.format(query=sample_query)

response = client.responses.create(
    model="gpt-6-luna",
    input=prompt_input
)

# expecting "roofing"
print(response.output_text)

roofing


In [10]:
import pandas as pd

queries = pd.read_csv('queries.txt', sep='\t')

prompt_template = "Categorize homeowner's request/issue by required service type to resolve them. Explicitly respond with the best service out of ONLY these three options: electrical, plumbing, roofing.\nQuery: {query}"
print(f'Prompt: {prompt_template}')
print(f'Prompt Character Length: {len(prompt_template)}')

results = []
for idx, row in queries.iterrows():
  query_id    = row['query_id']
  query_text  = row['query']
  category    = row['target_category']

  # format prompt
  prompt_input = prompt_template.format(query=query_text)

  # model response
  response = client.responses.create(
      model="gpt-6-luna",
      input=prompt_input
  )

  # ensure predicted category is cleaned (no white space, tabs, etc.)
  predicted_category = response.output_text.strip()

  results.append({
      'query_id': query_id,
      'predicted_category': predicted_category
  })

Prompt: Categorize homeowner's request/issue by required service type to resolve them. Explicitly respond with the best service out of ONLY these three options: electrical, plumbing, roofing.
Query: {query}
Prompt Character Length: 198


Save your prompt template in a file "short-prompt.txt".
Save the results of your prompt testing in "short-output.tsv" with columns `query_id` and `predicted_category`.

In [11]:
# save prompt template
with open('short-prompt.txt', 'w') as f:
  f.write(prompt_template)

# Save results
output_results = pd.DataFrame(results)
output_results.to_csv('short-outputs.tsv', sep='\t', index=False)
output_results.head(10)


,query_id,predicted_category
0,1,roofing
1,2,plumbing
2,3,electrical
3,4,roofing
4,5,plumbing
5,6,electrical
6,7,roofing
7,8,plumbing
8,9,electrical
9,10,roofing


Submit "short-prompt.txt" and "short-output.tsv" in Gradescope.

Hint: your prompt may be re-tested with the Gemini API, so do not rely solely on lucky language model responses.

## Part 2: Find Short Prompt Mistakes

Construct 5 queries of 100 characters or less that trick your short prompt so that the wrong category is chosen.


In [13]:
import pandas as pd
prompt_template = "Categorize homeowner's request/issue by required service type to resolve them. Explicitly respond with the best service out of ONLY these three options: electrical, plumbing, roofing.\nQuery: {query}"

trick_query_1 = "Washing machine doesn't fill up. Basement flooded 3 days ago but its dry and no leaks."  # expected electricity
trick_query_2 = "electrical poles fell over after a storm. upstairs bathroom walls are leaking by the shower" # expected plumbing
trick_query_3 = "branch hit my shingles. garbage disposal makes loud buzzing noise and won't turn on"   # expected electrical
trick_query_4 = "strong blizzard piled snow on roof and near house walls. its 60 out and garage door is disabled" # expected electrical
trick_query_5 = "roof is intact. Massive thunderstorm and water seeps in the house" # expected plumbing

edge_case_test = [
    {
        'query_id': 1,
        'query': trick_query_1,
        'target_category': 'electrical'
    },
    {
        'query_id': 2,
        'query': trick_query_2,
        'target_category': 'plumbing'
    },
    {
        'query_id': 3,
        'query': trick_query_3,
        'target_category': 'electrical'
    },
    {
        'query_id': 4,
        'query': trick_query_4,
        'target_category': 'electrical'
    },
    {
        'query_id': 5,
        'query': trick_query_5,
        'target_category': 'plumbing'
    }
]

edge_case_df = pd.DataFrame(edge_case_test)

edge_case_results = []
for idx, row in edge_case_df.iterrows():
  query_id    = row['query_id']
  query_text  = row['query']
  category    = row['target_category']

  # format prompt
  prompt_input = prompt_template.format(query=query_text)

  # model response
  response = client.responses.create(
      model="gpt-6-luna",
      input=prompt_input
  )

  # ensure predicted category is cleaned (no white space, tabs, etc.)
  predicted_category = response.output_text.strip()

  edge_case_results.append({
      'query_id': query_id,
      'query': query_text,
      'target_category': category,
      'predicted_category': predicted_category
  })

edge_case_outputs = pd.DataFrame(edge_case_results)
edge_case_outputs

,query_id,query,target_category,predicted_category
0,1,Washing machine doesn't fill up. Basement floo...,electrical,plumbing
1,2,electrical poles fell over after a storm. upst...,plumbing,electrical and plumbing
2,3,branch hit my shingles. garbage disposal makes...,electrical,Roofing and plumbing.
3,4,strong blizzard piled snow on roof and near ho...,electrical,roofing
4,5,roof is intact. Massive thunderstorm and water...,plumbing,roofing


Save your 5 queries in a file "mistakes.tsv" with columns `query`, `target_category` and `predicted_category`.

In [15]:
edge_case_outputs[['query', 'target_category', 'predicted_category']].to_csv('mistakes.tsv', sep='\t', index=False)

Submit "mistakes.tsv" in Gradescope.

## Part 3: Design a Long Prompt

Repeat part 1 with a length limit of 5000 characters.

In [ ]:
# YOUR CHANGES HERE

...

Save your longer prompt template in a file "long-prompt.txt".
Save the results of your prompt testing in "long-output.tsv".
Both files should use the same columns as part 1.

In [ ]:
# YOUR CHANGES HERE

...

Submit "long-prompt.txt" and "long-output.tsv" in Gradescope.

## Part 4: Use the Decisions API

In September 2026, TypeSafe AI announced [Jev](https://typesafe.ai/blog/introducing-system-one-models-and-jev), a new variant of large language models focused on decision making.
In October 2026, OpenAI introduced a similar API that they called a [Decisions API](https://openai.com/index/decisions-api/).
Use the Decisions API ([docs](https://developers.openai.com/api/docs/guides/decisions)) to the implement the same decisions.
Save the results of your testing to "decision-output.tsv" with the same columns as parts 1 and 3.

In [ ]:
# YOUR CHANGES HERE

...

## Part 5: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 6: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.